# Bibliotecas

In [225]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.cluster import KMeans
import numpy as np
from scipy.spatial.distance import cdist

# Importando os dados. 

In [227]:
caminho_parquet = r"C:\Users\Pichau\INMET\TODAS_ESTACOES\TODAS_ESTACOES_CONCATENADO.parquet"

df = pd.read_parquet(caminho_parquet)


df= df.drop(columns=['Dt_Hr'])
df= df.drop(columns=['timestamp'])
df= df.drop(columns=['Lat'])
df= df.drop(columns=['Long'])
df= df.drop(columns=['Alt'])

# Não quero explicar a variavel com a variavel 
df= df.drop(columns=['PRECIPITAÇÃO TOTAL primeira hora anterior'])
df= df.drop(columns=['PRECIPITAÇÃO TOTAL segunda hora anterior'])
df= df.drop(columns=['PRECIPITAÇÃO TOTAL terceira hora anterior'])

# Discretização: chuva

In [229]:
df_intermediario = df.copy()

In [230]:
condicoes = [
    df_intermediario['Precip'] == 0,
    (df_intermediario['Precip'] > 0) & (df_intermediario['Precip'] < 25),
    df_intermediario['Precip'] >= 25
]

valores = ['Sem chuva', 'Chuva', 'Chuva Extrema']

df_intermediario['CHUVA'] = np.select(condicoes, valores, default='Desconhecido')

In [231]:
# Verificando a quantidade de Desconhecidos. Ausente ou - 9999.0 

# Contagem absoluta
contagem_absoluta = df_intermediario['CHUVA'].value_counts()

# Contagem relativa (proporção)
contagem_relativa = df_intermediario['CHUVA'].value_counts(normalize=True)

# Exibir os resultados
print("Contagem Absoluta:")
print(contagem_absoluta)

print("\nContagem Relativa:")
print(contagem_relativa)

Contagem Absoluta:
CHUVA
Sem chuva        492347
Chuva             43505
Desconhecido      29821
Chuva Extrema       199
Name: count, dtype: int64

Contagem Relativa:
CHUVA
Sem chuva        0.870068
Chuva            0.076881
Desconhecido     0.052699
Chuva Extrema    0.000352
Name: proportion, dtype: float64


In [232]:
df_intermediario = df_intermediario[df_intermediario['CHUVA'] != 'Desconhecido']

# Discretização: VARIACAO PRESSAO

In [ ]:
def aplicar_kmeans_pressao(df, colunas_variacao, n_clusters=3, random_state=10):
    """
    Aplica K-Means à primeira coluna de variação de pressão e usa os centróides para rotular as demais.
    
    Parâmetros:
    - df: DataFrame
    - colunas_variacao: lista com os nomes das colunas de variação de pressão (em ordem)
    - n_clusters: número de clusters
    - random_state: para reprodutibilidade do K-Means
    
    Retorna:
    - DataFrame com colunas de cluster e rótulos adicionadas
    """
    total_inicial = len(df)
    df = df.dropna(subset=colunas_variacao).copy()  # <- CÓPIA EXPLÍCITA
    total_final = len(df)
    removidos = total_inicial - total_final
    percentual_removido = removidos / total_inicial * 100

    print(f"Total de linhas removidas por valores ausentes: {removidos} ({percentual_removido:.2f}%)")

    # Aplica KMeans na primeira coluna
    X = df[[colunas_variacao[0]]]
    kmeans = KMeans(n_clusters=n_clusters, random_state=random_state)
    df.loc[:, f'{colunas_variacao[0]} CLUSTER K MEANS'] = kmeans.fit_predict(X)
    
    # Obtém centróides
    centroids = kmeans.cluster_centers_
    print(f"Centróides de '{colunas_variacao[0]}':", centroids.flatten())

    # Rótulos com base nos índices dos centróides ordenados
    ordem_centroids = np.argsort(centroids.flatten())
    rotulos = ['Baixo', 'Medio', 'Alto']
    mapeamento = {ordem_centroids[i]: rotulos[i] for i in range(n_clusters)}

    df.loc[:, f'{colunas_variacao[0]} CLUSTER K MEANS - ROTULOS'] = df[f'{colunas_variacao[0]} CLUSTER K MEANS'].map(mapeamento)

    # Aplica os centróides manualmente nas colunas seguintes
    for coluna in colunas_variacao[1:]:
        X_novo = df[[coluna]]
        df.loc[:, f'{coluna} CLUSTER K MEANS'] = np.argmin(cdist(X_novo, centroids), axis=1)
        df.loc[:, f'{coluna} CLUSTER K MEANS - ROTULOS'] = df[f'{coluna} CLUSTER K MEANS'].map(mapeamento)

    return df


In [245]:
colunas_pressao = [
    'VARIACAO PRIMEIRA HORA ANTERIOR PRESSAO',
    'VARIACAO SEGUNDA HORA ANTERIOR PRESSAO',
    'VARIACAO TERCEIRA HORA ANTERIOR PRESSAO'
]

df_intermediario = aplicar_kmeans_pressao(df_intermediario, colunas_pressao)


Total de linhas removidas por valores ausentes: 1022 (0.19%)
Centróides de 'VARIACAO PRIMEIRA HORA ANTERIOR PRESSAO': [0.16660775 0.80650865 0.39217988]


In [283]:
df_discretizado = df_intermediario[[
    'CHUVA',
    'VARIACAO PRIMEIRA HORA ANTERIOR PRESSAO CLUSTER K MEANS - ROTULOS',
    'VARIACAO SEGUNDA HORA ANTERIOR PRESSAO CLUSTER K MEANS - ROTULOS',
    'VARIACAO TERCEIRA HORA ANTERIOR PRESSAO CLUSTER K MEANS - ROTULOS',

]]


# Export dados discretizados

In [288]:
caminho = r"C:\Users\Pichau\INMET\TODAS_ESTACOES\df_discretizado.csv"

df_discretizado.to_csv(caminho, index=False)
